# ClassTwin (SCIC 2026) - Kaggle End-to-End Pipeline
Bản sao số Lớp học: Tiền xử lý dữ liệu lớn EdNet, huấn luyện GRU-DKT, mô phỏng phản thực và tối ưu hóa chính sách sư phạm bằng Discrete Offline RL (CQL).

In [ ]:
# 1. Kiểm tra tài nguyên GPU và thư mục dữ liệu trên Kaggle
!nvidia-smi
!ls -lh /kaggle/input

In [ ]:
# 2. Cấu hình thư mục làm việc và kiểm tra mã nguồn
import os, sys
from pathlib import Path

# Giả sử repo nằm tại /kaggle/working hoặc được clone về
WORK_DIR = Path("/kaggle/working/classtwin")
if not WORK_DIR.exists():
    # Tạo symlink hoặc clone nếu cần
    print("Vui lòng clone hoặc nén mã nguồn classtwin vào /kaggle/working/")
sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / "src"))

In [ ]:
# 3. Chạy toàn bộ bộ kiểm thử tự động (13 tests)
!python -c "
import sys, importlib.util
from pathlib import Path
for tf in ['tests/test_schemas.py', 'tests/test_state_estimation.py', 'tests/test_twin.py', 'tests/test_cql.py', 'tests/test_inference.py']:
    spec = importlib.util.spec_from_file_location(Path(tf).stem, tf)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    for a in dir(mod):
        if a.startswith('test_'): getattr(mod, a)()
print('All 13 unit tests passed successfully!')
"

In [ ]:
# 4. Chạy luồng Pipeline Hợp nhất (Tiền xử lý toàn bộ archive + Huấn luyện Estimator + Huấn luyện Policy CQL)
!python scripts/pipeline.py \
    --config configs/kaggle_pipeline.yaml \
    --output-dir /kaggle/working/output \
    --estimator-epochs 10 \
    --policy-episodes 30 \
    --policy-steps 100

In [ ]:
# 5. Kiểm tra Release Bundle và Checkpoints đã sinh
import json
manifest_path = Path("/kaggle/working/output/release_manifest.json")
if manifest_path.exists():
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    print(json.dumps(manifest, indent=2))
else:
    print("Manifest chưa được tạo.")

!ls -lh /kaggle/working/output/models

In [ ]:
# 6. Thử nghiệm suy luận gợi ý sư phạm trực tuyến (Decision Support)
from src.classtwin.inference.state_store import ClassroomStateStore
from src.classtwin.inference.engine import DecisionSupportEngine

store = ClassroomStateStore()
engine = DecisionSupportEngine(state_store=store)

# Giả lập một lớp học có 40 sinh viên đang hoạt động
import numpy as np
for i in range(40):
    store.update_student_state(
        class_id="class_kaggle_demo",
        student_id=f"u_{i}",
        hidden=np.random.randn(64).astype(np.float32),
        ts_ms=1700000000 + i,
        mastery=float(np.random.beta(2, 2))
    )

rec = engine.generate_recommendation(
    class_id="class_kaggle_demo",
    topic="machine_learning",
    time_remaining_minutes=35.0,
    run_simulation_rollout=True
)
print("=== GỢI Ý QUYẾT ĐỊNH CHO GIẢNG VIÊN ===")
print(json.dumps(rec, indent=2, ensure_ascii=False))